# 🫧 K-Means Clustering
**ISLP Chapter 12 · Pattern Recognition for the Rest of Us**

> K-means partitions observations into K clusters so that within-cluster variation is minimized. It's the most widely used clustering method — fast, scalable, and produces interpretable results. The hard part is choosing K and interpreting what the clusters mean.

### What you'll learn
- Lloyd's algorithm: how K-means iterates to convergence
- The elbow method and silhouette score for choosing K
- Sensitivity to initialization — k-means++ fix
- Visualizing clusters with PCA
- Real-world application: customer segmentation

### Dataset: Simulated + Mall Customer Segmentation
### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore')
print('✓ Libraries loaded')

## 🎯 Part 1 — Lloyd's Algorithm

K-means minimizes the total within-cluster sum of squares (WCSS):
```
minimize Σₖ Σᵢ∈Cₖ ||xᵢ - μₖ||²
```

**Algorithm:**
1. Randomly initialize K cluster centroids
2. **Assignment step:** assign each point to its nearest centroid
3. **Update step:** recompute each centroid as the mean of its assigned points
4. Repeat steps 2–3 until assignments stop changing

K-means is guaranteed to converge but may find a local (not global) minimum. Run it multiple times (n_init=10 in sklearn by default) and keep the best solution.

In [ ]:
# Animate K-means convergence step by step
np.random.seed(42)
# Generate 3 clusters
X_demo = np.r_[
    np.random.multivariate_normal([0,0], [[1,0],[0,1]], 50),
    np.random.multivariate_normal([5,1], [[1,0.5],[0.5,1]], 50),
    np.random.multivariate_normal([2,5], [[0.8,0],[0,0.8]], 50),
]

# Show 4 iterations manually
colors_k = ['#1e5fa8','#e85d20','#1a7a45']
np.random.seed(0)
centroids = X_demo[np.random.choice(len(X_demo), 3, replace=False)]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for step in range(6):
    labels = np.argmin(np.linalg.norm(X_demo[:,None] - centroids[None,:], axis=2), axis=1)
    ax = axes[step]
    for k in range(3):
        mask = labels==k
        ax.scatter(X_demo[mask,0], X_demo[mask,1], color=colors_k[k], alpha=0.6, s=20)
        ax.scatter(*centroids[k], color=colors_k[k], s=200, marker='*',
                  edgecolors='black', lw=1.5, zorder=5)
    ax.set_title(f'Iteration {step+1} — Assignment' if step%2==0 else f'Iteration {step+1} — Update')
    # Update centroids
    new_centroids = np.array([X_demo[labels==k].mean(axis=0) for k in range(3)])
    if step%2==1 and step<5:  # draw movement lines
        for k in range(3):
            ax.annotate('', xy=new_centroids[k], xytext=centroids[k],
                       arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    centroids = new_centroids

plt.suptitle('K-Means: Step-by-Step Convergence (K=3)', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Elbow + Silhouette
wcss = []
sil_scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_mall_s)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_mall_s, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), wcss, 'o-', color='#1e5fa8', lw=2, markersize=7)
best_k_sil = list(K_range)[np.argmax(sil_scores)]
axes[0].set_xlabel('K'); axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method — look for the bend')

axes[1].plot(list(K_range), sil_scores, 's-', color='#e85d20', lw=2, markersize=7)
axes[1].axvline(best_k_sil, color='#1a7a45', lw=1.5, ls='--', label=f'Best K={best_k_sil}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher is better')
axes[1].legend()
plt.suptitle('Choosing K — Elbow and Silhouette Methods', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print(f'\n📌 Silhouette picks K={best_k_sil} as optimal')
print('   Elbow bend should also be visible — look for where WCSS stops dropping steeply')

In [ ]:
# Final clustering + visualization
best_k = best_k_sil
km_final = KMeans(n_clusters=best_k, n_init=20, random_state=42)
labels_final = km_final.fit_predict(X_mall_s)
centroids_orig = sc.inverse_transform(km_final.cluster_centers_)

fig, ax = plt.subplots(figsize=(9, 6))
for k in range(best_k):
    mask = labels_final == k
    ax.scatter(X_mall[mask,0], X_mall[mask,1], label=f'Cluster {k+1} (n={mask.sum()})',
              s=40, alpha=0.7, edgecolors='white', lw=0.5)
    ax.scatter(*centroids_orig[k], s=250, marker='*', edgecolors='black', lw=1.5, zorder=5,
              color=plt.gca().lines[-1].get_color() if plt.gca().lines else 'red')

ax.set_xlabel('Annual Income (k$)'); ax.set_ylabel('Spending Score')
ax.set_title(f'K-Means Customer Segmentation (K={best_k})')
ax.legend()
plt.tight_layout(); plt.show()

# Cluster profiles
profile = customers.copy()
profile['Cluster'] = labels_final
print('\n=== Cluster Profiles ===')
print(profile.groupby('Cluster')[['Age','Annual Income (k$)','Spending Score (1-100)']].mean().round(1))

In [ ]:
answers = {
    'q1': '',  # What does K-means minimize?
    'q2': '',  # Why should you run K-means multiple times with different initializations?
    'q3': '',  # What does a silhouette score close to 1.0 indicate?
    'q4': '',  # What is one limitation of K-means clustering?
    'q5': '',  # K-means assumes clusters are what shape?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

## 📚 Further Reading
- [ISLP Ch 12.4](https://intro-stat-learning.github.io/ISLP/) — K-Means
- [Hierarchical Clustering notebook](hierarchical-clustering.ipynb) — alternative without choosing K upfront
- [PCA notebook](pca.ipynb) — dimensionality reduction before clustering

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*